# Solutions · Chapter 01-06 · Charts and reproducible randomness

Worked answers with reasoning. E5 and E10 are about the same thing from two directions: how to be
misleading without saying anything false, and why "we didn't change any data" is not a defence.

Self-contained: run from the top with a fresh kernel.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

months = pd.date_range("2023-01-01", periods=12, freq="MS")
rentals = np.array([1040, 1010, 950, 1005, 970, 1020, 960, 995, 940, 980, 1030, 1060])
monthly = pd.Series(rentals, index=months, name="rentals")

rng = np.random.default_rng(0)
n = 60
X = rng.normal(0, 1, (n, 3))
y = 2 * X[:, 0] - X[:, 1] + rng.normal(0, 2, n)
print("data ready")

## E1 · The two techniques

**A truncated y-axis.** Starting at 900 instead of 0 turned a 13% change into most of the plot's
height. It works because the reader interprets *height* as *magnitude*, and height is now set by
the axis rather than by the data.

**A chosen window.** Showing four months out of twelve, selected because they rise. It works
because a chart shows what is on it and gives no hint about what is not - the reader has no way to
know that a choice was made.

**Why the pair is so effective:** each is individually defensible. "We're focusing on recent
performance" and "the axis shows the relevant range" are both sentences that survive a meeting.
Together they produce a conclusion the full data does not support.

## E2 · When truncation is legitimate

**Required:** body temperature. Plotted from 0 °C, the clinically enormous difference between 36.5
and 39.5 is a barely visible wiggle at the top of the chart. Everything of interest lives in a
narrow band and the axis must show that band.

**Dishonest:** the chapter's chart. Rentals genuinely could be near zero - a closed month is a real
possibility - and the message is *how much* they changed, so the zero point is meaningful and
removing it exaggerates.

**The rule:** start at zero when the length or height *is* the message. Truncate when the variation
is the message and zero is not a meaningful reference - and make the truncation obvious, with a
clearly labelled axis, a broken-axis marker, or a stated range in the title.

**The test I actually apply:** would a reader who glanced at the shape and read nothing else reach
the same conclusion as a reader who studied the numbers? If not, the axis is doing work the data
is not.

## E3 · `default_rng` versus the global seed

`np.random.seed(0)` sets **one hidden generator shared by the entire process**. Any code that draws
from it - a library function, a helper, a cell you ran ten minutes ago - advances its state. So your
results depend on *what else has run and in what order*, which is invisible and changes every time
someone re-runs a cell.

`np.random.default_rng(0)` returns **your own generator object**. Nothing else can advance it, and
you can pass it explicitly to whatever needs randomness, which makes the dependency visible in the
code.

**The practical consequence:** a notebook seeded globally is reproducible when run top to bottom and
irreproducible the moment anyone re-runs a cell out of order - which is what notebooks are for.
That is E9.

## E4 · Three true numbers, three different stories

In [ ]:
first_to_last = rentals[-1] / rentals[0] - 1
ninth_to_last = rentals[-1] / rentals[8] - 1
half_difference = rentals[6:].mean() - rentals[:6].mean()

print(f"first month to last month : {first_to_last:+.1%}")
print(f"ninth month to last month : {ninth_to_last:+.1%}")
print(f"second half minus first half: {half_difference:+.1f} rentals per month "
      f"({half_difference / rentals[:6].mean():+.1%})")

| Measure | Value | Story it tells |
|---|---|---|
| First to last month | **+1.9%** | Essentially flat |
| Ninth to last month | **+12.8%** | Strong growth |
| Second half minus first half | **-5.0 rentals (-0.5%)** | Slightly declining |

All three are correct arithmetic on the same twelve numbers.

**Which would I headline? The half-to-half comparison, or better, none of them alone.** It uses
every observation, it is not sensitive to which single month you start and end on, and it gives the
answer Maria actually asked for: the stand is not getting busier.

**What must be said alongside it:** that twelve months is one seasonal cycle, so a rise in
November and December cannot be separated from "November and December are busy months". You would
need a second year to distinguish a trend from a season - which is the limitation module 09 keeps
returning to, and the same limitation `data/README.md` records for the Seoul bike dataset.

**The general lesson:** any measure that depends on two endpoints - first to last, this month
versus last month, year on year - can be moved a long way by choosing the endpoints. Measures that
use all the data are much harder to manipulate, deliberately or accidentally.

## E5 · Honest and dishonest, both true

**Honest:**

> "MAE was 1.47 on one split. Across twenty splits it ranged from 1.24 to 1.92, so a single number
> overstates how precisely we know this - treat it as roughly 1.5, give or take 0.3."

**Dishonest, without stating anything false:**

> "The model achieves an MAE of 1.24."

Every word of the second is true. It reports a measured value, on real held-out data, with no
alteration. It is dishonest because it presents the **best** of twenty draws as though it were the
result, and omits the fact that there were twenty.

**The distinction is not about accuracy but about what was left out.** This is the same technique as
the chosen window in the chart, and it is why 07-03 insists on reporting the spread and 07-04 on
never selecting on the test set.

**The habit:** whenever you write down a single number, ask *how many numbers did I have to choose
from to get this one?* If the answer is more than one, say so.

## E6 · The chart, drawn honestly and truncated legitimately

In [ ]:
fig, (honest, zoomed) = plt.subplots(1, 2, figsize=(11, 4))

honest.plot(monthly.index, monthly.values, marker="o", color="#0072B2")
honest.axhline(monthly.mean(), color="grey", linestyle="--", linewidth=1,
               label=f"year mean {monthly.mean():.0f}")
honest.set_ylim(0, 1200)
honest.set_xlabel("Month"); honest.set_ylabel("Bikes rented per month (count)")
honest.set_title("Rentals were flat across 2023 (+1.9% first to last month)")
honest.legend()

zoomed.plot(monthly.index, monthly.values, marker="o", color="#0072B2")
zoomed.set_ylim(900, 1100)
zoomed.set_xlabel("Month"); zoomed.set_ylabel("Bikes rented per month (count)")
zoomed.set_title("Same data, zoomed to 900-1100 to show month-to-month variation")
zoomed.text(monthly.index[0], 1085, "note: y-axis does not start at zero",
            fontsize=8, color="#D55E00")

fig.tight_layout()
plt.show()

The right-hand chart truncates the axis and is not misleading, because three things make the
truncation impossible to miss: the title says what the range is, an annotation states it in words,
and the title describes the chart's purpose as showing *variation* rather than *level*.

**The difference between this and the failure lab is not the axis limits. It is the disclosure.**
The same `set_ylim` call is dishonest with a title that says "up 13%" and honest with a title that
says "zoomed to show variation".

Note also what the honest chart adds that neither original had: **a reference line at the mean**.
A single line of code, and it converts "some wiggly points" into "months above and below average" -
the reader can now see the answer instead of estimating it. Adding a reference - a mean, a target,
a baseline, last year - is the cheapest improvement available to almost any chart.

## E7 · Comparing two models properly

In [ ]:
def compare_models(X, y, models, n_splits=200, test_size=0.35, seed=0):
    """MAE distribution for each model across many random splits, plus how often each wins."""
    gen = np.random.default_rng(seed)                  # seeded, so this function is reproducible
    scores = {name: [] for name in models}
    for _ in range(n_splits):
        split_seed = int(gen.integers(0, 2**31 - 1))   # a different split each time, all repeatable
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=test_size, random_state=split_seed)
        for name, model in models.items():
            scores[name].append(mean_absolute_error(y_te, model.fit(X_tr, y_tr).predict(X_te)))

    frame = pd.DataFrame(scores)
    summary = frame.agg(["mean", "std", "min", "max"]).T.round(3)
    names = list(models)
    summary["win_rate"] = [ (frame[n] == frame[names].min(axis=1)).mean().round(3) for n in names ]
    return summary


summary = compare_models(X, y, {"linear": LinearRegression(), "ridge": Ridge(alpha=3.0)})
print(summary)

**The sentence I would put in a report:**

> "Across 200 random splits, ridge averaged 1.749 MAE against linear's 1.765 and won 67% of
> splits. That 0.016 gap is tiny next to the split-to-split spread, which ran from 1.15 to 2.50
> with a standard deviation of about 0.23. On this data the two models are indistinguishable; the
> choice should be made on other grounds."

Three deliberate features of that function, and all three are the point:

- **It is seeded**, so the comparison itself is reproducible - but it draws a *different* split each
  iteration from that seeded generator. Both properties are needed: vary the split, fix the
  experiment.
- **It reports a distribution, not a number.** Mean, spread, and range, so the reader can see
  whether a difference is meaningful before being told that it is.
- **The win rate is separate from the mean.** A model can win most splits by a hair and lose badly
  on a few; the two numbers together tell you which.

This is a hand-rolled version of what `cross_val_score` and `RepeatedKFold` do properly in 04-07 and
07-03 - including the part this version skips, which is that repeated random splits are not the same
as k-fold, and overlapping test sets make the standard deviation an optimistic estimate of the real
uncertainty.

## E8 · Five bins

**What I would check:** redraw it with 15, 30 and 60 bins, and also draw the raw values as a strip
or a box plot. A shape that survives every bin count is probably real; a shape that appears at one
bin count and vanishes at another was made by the binning.

**What 5 bins can hide:**

- **Bimodality.** Two peaks - weekdays and weekends, say - merge into one broad hump that looks
  perfectly normal. This is the most common and most consequential thing 5 bins destroys, because
  two populations in one column changes what you should model.
- **A spike at one value.** A pile of zeros or a sentinel like `-999` disappears into a wide bin
  and looks like part of a smooth distribution.
- **The tail.** A handful of extreme days sit in the last bin with everything else near them, so a
  long right tail looks like a short one - and skew is exactly what decides whether you should
  transform the target (02-05).
- **Gaps.** Values that never occur - a rounding artefact, a capped instrument - are invisible when
  bins are wide.

**And the deeper point:** "roughly normal" is a claim that usually does not matter. Very few methods
in this course require normally distributed *data* - the assumptions, where they exist, are about
residuals or errors. The useful questions about a distribution are "is it one population or
several?", "is it skewed?", "is there a floor or a ceiling?", and "are there impossible values?".
02-05 asks those properly.

## E9 · Reproducible top to bottom, irreproducible on re-run

**What is happening.** `np.random.seed(0)` sets the global generator once, at the top. Every draw
after that advances it. Run the notebook top to bottom and the modelling cell always sees the
generator in the same state, because the same sequence of draws preceded it. Re-run the modelling
cell alone and the generator is wherever the previous run left it - so you get the *next* numbers
in the sequence, not the same ones.

The result is a notebook that appears reproducible under exactly the conditions where nobody
notices, and stops being reproducible under exactly the conditions where people work.

**The fix, in order of preference:**

1. **Create a generator where the randomness is used.** `rng = np.random.default_rng(0)` inside the
   modelling cell - or better, passed into the function that needs it. The cell then produces the
   same result no matter what ran before it.
2. **Set `random_state=` explicitly on every estimator, splitter and sampler.** scikit-learn objects
   fall back to the global generator when you do not, which is the same trap.
3. **Never rely on cell order for correctness.** If a notebook only works top to bottom, say so -
   and consider whether the logic belongs in a function or a script instead, which is 04-08's
   argument.

**The diagnostic that finds it in ten seconds:** run the suspect cell twice in a row. If the answer
changes, something in it is drawing from an unseeded or shared generator.

## E10 · "We reported the best of five seeds"

> Choosing the best of five seeds is choosing the test set that flatters the model, and that is a
> decision about the data even though no data was altered. What you have reported is not the
> model's performance but the maximum of five noisy draws, which is biased upward by an amount that
> grows with how many seeds you tried and how noisy the estimate is - and neither of those is
> stated. The honest version costs nothing: run twenty seeds and report the mean and the range, which
> is a *better* result because it also tells the reader how precisely you know it. If the model only
> looks good on one seed in five, that is the finding, and it is one the next person deserves to
> hear.

**What is being tested:** whether you recognise selection as a modification. It is the same move as
the chosen window in E1 and the "MAE of 1.24" in E5 - nothing false is said, and the number no
longer means what it appears to mean. Chapter 07-04 shows how large the resulting optimism can get.

## E11 · One chart for Maria

**The chart:** a line chart of all twelve months of 2023, y-axis from zero, x-axis labelled by
month, y-axis labelled "Bikes rented per month (count)". A dashed horizontal line at the year's
mean. Title: "Monthly rentals, 2023 - flat across the year".

If a second year of data existed I would put it on the same axes as a second line, because that is
the only way to separate "December is busy" from "the stand is growing".

**The sentence I would say:**

> "Rentals bounce around by five or ten percent from month to month, but the year ends where it
> started - the last two months are up, and that is what December usually does, so I would not call
> it growth until we see next spring."

**Why that chart and not a prettier one.** Maria's decision is whether to invest in more bikes. The
chart has to make "flat" and "not flat" visually distinguishable, and it must not let a two-month
uptick look like a trend. Everything else - colour, style, a moving average - is decoration on that
job. **Design the chart for the decision, not for the data.**

## E12 · "Average waiting time: 24 minutes"

**Chart 1: a histogram of individual waiting times.** The average hides the shape entirely. Twenty-
four minutes could be almost everyone waiting 20-30, or half the patients seen in five minutes and
half waiting an hour. Those demand completely different responses, and only the distribution
distinguishes them. It would also reveal a long tail - the patients who waited three hours, who are
the ones that generate complaints and harm, and who barely move the mean.

**Chart 2: waiting time by hour of day (or day of week), as a line or a box per hour.** An average
over the week hides the pattern that is actionable. If Monday mornings run at 50 minutes and
Thursday afternoons at 8, the answer is staffing, not capacity - and the single number cannot
distinguish a system that is uniformly slow from one that is badly scheduled.

**What both have in common:** they replace a summary with a *shape*, and shapes support decisions
that summaries cannot. The general form is worth stating, because it recurs throughout this course:
**one number tells you the level; you almost always need the spread and the segments to know what
to do.** It is the same argument as error analysis by slice in 07-05 - a model's overall score is
the hospital's 24 minutes.

## E13 · Explaining it to Maria

> Every number on that chart was real. But it only showed the last four months, and it started the
> scale at 900 instead of zero - so a small rise filled the whole picture. Show all twelve months
> from zero and the line is flat. Nothing was faked; the chart was just answering a narrower
> question than the one you asked.

(62 words.)

**Why this matters for how you work rather than just how you present:** the same two moves are how
you fool *yourself*. You zoom in to see something clearly, then keep looking at the zoomed version
and forget the scale. Redrawing your own charts from zero, with the full range, before you believe
them, is the cheapest defence there is.

## E14 · How much of a reported score is noise?

In [ ]:
gen = np.random.default_rng(1)
ridge_scores = []
for _ in range(500):
    split_seed = int(gen.integers(0, 2**31 - 1))
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.35, random_state=split_seed)
    ridge_scores.append(mean_absolute_error(y_te, Ridge(alpha=3.0).fit(X_tr, y_tr).predict(X_te)))
ridge_scores = np.array(ridge_scores)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.35, random_state=3)
single = mean_absolute_error(y_te, Ridge(alpha=3.0).fit(X_tr, y_tr).predict(X_te))
percentile = (ridge_scores < single).mean() * 100

print(f"across 500 splits: mean {ridge_scores.mean():.3f}, sd {ridge_scores.std():.3f}, "
      f"range {ridge_scores.min():.3f} to {ridge_scores.max():.3f}")
print(f"the single random_state=3 score: {single:.3f}  ->  {percentile:.0f}th percentile")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.hist(ridge_scores, bins=35, color="#0072B2", edgecolor="white")
ax.axvline(single, color="#D55E00", linewidth=2.5, label=f"random_state=3: {single:.2f}")
ax.axvline(ridge_scores.mean(), color="grey", linestyle="--", linewidth=1.5,
           label=f"mean of 500 splits: {ridge_scores.mean():.2f}")
ax.set_xlabel("MAE on the held-out rows")
ax.set_ylabel("Number of splits")
ax.set_title("One split is one sample from this distribution")
ax.legend()
plt.show()

**The single split that `random_state=3` produced sits at the 8th percentile** - one of the most
flattering draws in five hundred, not a typical one. At 1.416 against a mean of 1.778 it understates
the model's error by 0.36 MAE, roughly a fifth, and nothing about it looks unusual.

**The sentence I would use to warn a colleague:**

> "That 1.42 is one draw from a distribution that runs from 1.15 to 2.56 with a mean of 1.78. It
> is not the model's score; it is the score of one split, and it sits in the best tenth of them.
> Report the mean and the range from repeated splits, or use cross-validation."

**Two things this histogram teaches that no single number can.**

First, the **spread is the honest error bar**. Anyone comparing two models whose means differ by
less than this spread is comparing noise, and the histogram makes that immediately visible instead
of arguable.

Second, the distribution is **wide because the dataset is small** - 60 rows, 21 of them in each test
set. With 6,000 rows it would be far narrower. So "how much can I trust one split?" is really a
question about sample size, and the answer changes as the data grows. Chapter 03-03 gives you the
tools to quantify this, and 07-01 replaces repeated random splits with cross-validation, which uses
each row exactly once and is a better estimator of the same thing.

---

## Where to go next

Back to the chapter for the mastery check and flashcards. That finishes **module 01**.

Next is **02-01 · What is a row?**, where the course proper resumes and the tools you have just
learned get pointed at a harder question: what does this data actually mean, and what is wrong with
it?